In [1]:
import pyrosetta
import numpy as np

from benchmark import bpti_ryfyn_benchmark, full_bpti_benchmark
from misc import init_generator_params
from rotamer_extraction import extract_top_n_rotamers, TrackedResidue
from h_ising_creation import extract_hamiltonian_tensors, build_ising_hamiltonian, reduce_hamiltonian
from initialisation import initialize_rosetta
from custom_qaoa import qaoa_func_generator, run_qaoa
from h_mixer import custom_xy_mixer_layer

from constants import *
from validation import validate_conformations

initialize_rosetta(pyrosetta, extra_flags="-mute all")

Initializing PyRosetta with cleaning flags: -ignore_unrecognized_res and extra flags: -mute all
┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python313.m1 2026.03+releasequarterly.5e498f1409c68ade56c8ce5842bf79e1b02e8db4 2026-01-13T13:24:11] retrieved from: http://www.pyrosetta.org


In [2]:
# Pyrosetta Relevant Code
benchmark_pose = full_bpti_benchmark()
rotamer_lib, ig, rot_sets, scorefxn = extract_top_n_rotamers(benchmark_pose)

# Generating QUBO (Quadratic Unconstrained Binary Optimisation) Model, and then reduce it
# h_linear, J_quadratic = extract_hamiltonian_tensors(rotamer_lib, ig)
# h_flex_linear, J_flex_quadratic, global_offset = reduce_hamiltonian(h_linear, J_quadratic, rotamer_lib)

h_flex_linear, J_flex_quadratic = extract_hamiltonian_tensors(rotamer_lib, ig)
global_offset = 0

generator_params = init_generator_params(h_flex_linear)
# for idx in h_linear:
#     print(h_linear[idx])
#     print(h_flex_linear.get(idx, "None"))
#     print("\n---------------------------------------\n")

# Generate the actual observable and running functions we will use in the QAOA Algorithm
H_ising = build_ising_hamiltonian(h_flex_linear, J_flex_quadratic, global_offset)
cost_function, sample_function = qaoa_func_generator(H_ising, custom_xy_mixer_layer, generator_params)

# Run the Quantum Approximate Optimisation Algorithm and sample the final parameters
final_params = run_qaoa(cost_function)
probabilities = sample_function(final_params)

# Extract the top 100 most probably conformations and check that exactly 1 rotamer for each residue is selected
top_indices = list(np.argsort(probabilities)[-TOP_CONFORMATION_COUNTS:][::-1])
valid_conformations = validate_conformations(top_indices, probabilities, generator_params)


Creating score function
Pose scored successfully!
Creating Repacking Task - Core Rotamer Optimisation Protocol
Computing One-Body and Two-Body Energies
Iterating through molten residues - determining the top rotamer positions for each amino acid
Moltenres ID: 1, SeqPos ID: 20
Moltenres ID: 2, SeqPos ID: 21
Moltenres ID: 3, SeqPos ID: 22
Moltenres ID: 4, SeqPos ID: 23
Moltenres ID: 5, SeqPos ID: 24
initializing generator_params
Reduced Hamiltonian built: 15 Qubits, 70 Pauli strings.
Commencing QAOA Optimization [p=2]...
Epoch   0 | Cost: -25.4365
Epoch  10 | Cost: -25.4366
Epoch  20 | Cost: -25.4367
Epoch  30 | Cost: -25.4367
Epoch  40 | Cost: -25.4367
Epoch  50 | Cost: -25.4367
Epoch  60 | Cost: -25.4367
Epoch  70 | Cost: -25.4367
Epoch  80 | Cost: -25.4367
Epoch  90 | Cost: -25.4367
Epoch 100 | Cost: -25.4367
Epoch 110 | Cost: -25.4367
Epoch 120 | Cost: -25.4367
Epoch 130 | Cost: -25.4367
Epoch 140 | Cost: -25.4367
Optimization converged.
{20: 0, 21: 2, 22: 6, 23: 7, 24: 11}
Valid to 

In [3]:
global_offset

0

In [4]:
import importlib, energy_calculation, validation
importlib.reload(validation)
importlib.reload(energy_calculation)

from energy_calculation import calculate_and_compare_energies
calculate_and_compare_energies(valid_conformations, h_flex_linear, J_flex_quadratic, global_offset, benchmark_pose, scorefxn, rotamer_lib, generator_params)

==================== ENERGY OPERATIONS ====================
Calculating Quantum Energies for all 100 conformations
Calculating Pyrosetta for all 100 conformations 
Comparing both energy types for all 100 conformations 
The difference was too large to ignore for conformation 14.699632960404088: Conformation(bitstring=[1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0], probability=np.float64(0.9999999999736209), quantum_energy=np.float64(-25.43668570369482), biological_energy=np.float64(-40.13631866409891), pose=<pyrosetta.rosetta.core.pose.Pose object at 0x113a0c6b0>)
The difference was too large to ignore for conformation 14.699631829007572: Conformation(bitstring=[1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0], probability=np.float64(2.998657387264138e-12), quantum_energy=np.float64(-25.435701869428158), biological_energy=np.float64(-40.13533369843573), pose=<pyrosetta.rosetta.core.pose.Pose object at 0x1141e1cb0>)
The difference was too large to ignore for conformation 14.699632450836631: C

In [5]:
errors = [np.float64(14.699632960404088), np.float64(14.699631829007572), np.float64(14.699632450836631), np.float64(14.699633028729366), np.float64(14.694622256749199), np.float64(14.699631533414546), np.float64(14.694344762839734), np.float64(14.683775206342915), np.float64(14.67138652124018), np.float64(14.699632578438042), np.float64(14.699632541451109), np.float64(14.69963189733285), np.float64(14.69963251916191), np.float64(14.69462112535274), np.float64(14.694621627972452), np.float64(14.694622325074477), np.float64(14.699631601739881), np.float64(14.694620710550424), np.float64(14.694343631443274), np.float64(14.694344253272277), np.float64(14.694344831165012), np.float64(14.694343335850249), np.float64(14.683774074946399), np.float64(14.683774696775458), np.float64(14.683775274668193), np.float64(14.678764502688054), np.float64(14.683773779353345), np.float64(14.67848700877856), np.float64(14.67138538984372), np.float64(14.671385892463434), np.float64(14.671386589565458), np.float64(14.671384975041406), np.float64(14.655528767179007), np.float64(14.699631447041526), np.float64(14.699632068870613), np.float64(14.694621874783152), np.float64(14.699631151448585), np.float64(14.699631410054593), np.float64(14.69963203188368), np.float64(14.69462183779622), np.float64(14.699631114461653), np.float64(14.694344380873673), np.float64(14.69434434388674), np.float64(14.683774826131398), np.float64(14.683774789144465), np.float64(14.67138613927419), np.float64(14.671386102287258), np.float64(14.69462119367796), np.float64(14.69462169629773), np.float64(14.694620778875702), np.float64(14.694343699768496), np.float64(14.694344321597612), np.float64(14.69434340417547), np.float64(14.683774143271677), np.float64(14.68377476510068), np.float64(14.678763371291595), np.float64(14.678763873911294), np.float64(14.678764571013332), np.float64(14.683773847678566), np.float64(14.678762956489237), np.float64(14.678485877382101), np.float64(14.678486499211076), np.float64(14.678487077103782), np.float64(14.678485581788962), np.float64(14.671385458168999), np.float64(14.671385960788768), np.float64(14.671385043366683), np.float64(14.655527635782548), np.float64(14.655528138402232), np.float64(14.655528835504228), np.float64(14.655527220980119), np.float64(14.694620743386693), np.float64(14.69462124600642), np.float64(14.694620328584392), np.float64(14.69462070639976), np.float64(14.694621209019488), np.float64(14.69462029159746), np.float64(14.694343249477214), np.float64(14.694343871306302), np.float64(14.694342953884217), np.float64(14.694343212490281), np.float64(14.694343834319369), np.float64(14.694342916897284), np.float64(14.683773694734882), np.float64(14.683774316563856), np.float64(14.678764122476537), np.float64(14.683773399141828), np.float64(14.683773657747949), np.float64(14.683774279576923), np.float64(14.678764085489604), np.float64(14.683773362154895), np.float64(14.678486628566958), np.float64(14.678486591580025), np.float64(14.671385007877731), np.float64(14.671385510497458), np.float64(14.671384593075373), np.float64(14.671384970890799), np.float64(14.671385473510526), np.float64(14.67138455608844), np.float64(14.655528386967461)]

print(f"Mean {np.mean(errors)}, Stdev {np.std(errors)}, Max {np.max(errors)}, Min {np.min(errors)}")

Mean 14.685340311495485, Stdev 0.0122728968924223, Max 14.699633028729366, Min 14.655527220980119


In [8]:
from validation import Conformation
from rotamer_extraction import TrackedRotamer
import itertools

one_body = []
two_body = []

def evaluate_two_energies_alt(pose, valid_conformations: list[Conformation], scorefxn, residue_library: dict[int, TrackedResidue], params, ig, rot_sets, h_linear, J_quadratic):
    for conformation in valid_conformations:

        bitstring = conformation.bitstring
        bio_energy, _ = evaluate_singular_pyrosetta_energy_alt(pose, bitstring, scorefxn, residue_library, params)
        # conformation.biological_energy = bio_energy

        quant_energy = evaluate_quantum_energy_alt(bitstring, residue_library, params, ig, rot_sets, h_linear, J_quadratic)
        # conformation.quantum_energy = quant_energy
        print("Bio:", bio_energy, conformation.biological_energy, bio_energy - conformation.biological_energy)
        print("Qua:", quant_energy, conformation.quantum_energy, quant_energy - conformation.quantum_energy)
        print("Diff:", bio_energy-quant_energy)
        print()

    # print("==============ONE BODY ENERGIES==============")
    # print(one_body)
    #
    # print("==============TWO BODY ENERGIES==============")
    # print(two_body)


def evaluate_singular_pyrosetta_energy_alt(pose, bitstrings, scorefxn, residue_library: dict[int, TrackedResidue], params):
    test_pose = pose.clone()

    seq_positions = params["seq_positions"]
    wire_offsets = params["wire_offsets"]
    rotamer_counts = params["rotamer_counts"]

    for seq in seq_positions:
        base_wire = wire_offsets[seq]
        num_rots = rotamer_counts[seq]

        residue_bits = bitstrings[base_wire : base_wire + num_rots]
        local_rotamer_idx = residue_bits.index(1)

        res_entry: TrackedResidue = residue_library[seq]
        rotamer_entry = res_entry.rotamers[local_rotamer_idx]

        test_pose.replace_residue(seq, rotamer_entry.residue, False)

    bio_energy = scorefxn(test_pose)
    return bio_energy, test_pose

def evaluate_quantum_energy_alt(bitstring, residue_library: dict[int, TrackedResidue], params, ig, rot_sets, h_linear, J_quadratic):
    seq_positions = params["seq_positions"]
    seq_to_molten = {rot_sets.moltenres_2_resid(m): m for m in range(1, rot_sets.nmoltenres() + 1)}

    wire_offsets = params["wire_offsets"]
    rotamer_counts = params["rotamer_counts"]


    def get_picked_rotamer_idx(seq_idx):
        base_wire = wire_offsets[seq_idx]
        num_rots = rotamer_counts[seq_idx]

        residue_bits = bitstring[base_wire : base_wire + num_rots]
        local_rotamer_idx = residue_bits.index(1)

        rotamer_seq_entry = residue_library[seq_idx]
        rotamer_entry = rotamer_seq_entry.rotamers[local_rotamer_idx]

        return rotamer_entry.original_pyrosetta_index

    one_body_energies = 0
    for seq_idx in seq_positions:
        base_wire = wire_offsets[seq_idx]
        num_rots = rotamer_counts[seq_idx]

        residue_bits = bitstring[base_wire : base_wire + num_rots]
        local_rotamer_idx = residue_bits.index(1)

        single_energy = residue_library[seq_idx].rotamers[local_rotamer_idx].one_body_energy
        alt_single_energy = h_linear[seq_idx][local_rotamer_idx]

        assert single_energy == alt_single_energy, f"Energies do not match {seq_idx}{local_rotamer_idx}"

        one_body_energies += single_energy

    alt_one_body_energies = 0

    for seq, energies in h_linear.items():
            base_wire = wire_offsets[seq]
            # print(base_wire)
            for rot, e_val in energies.items():
                # print("\t", rot, "=>", base_wire+rot)
                if bitstring[base_wire + rot] == 1:
                    # print("\t\t", "Found item in bitstring", rot, e_val)
                    alt_one_body_energies += e_val
    one_body.append((one_body_energies, alt_one_body_energies, one_body_energies == alt_one_body_energies))

    two_body_energies = 0
    for seq_i, seq_j in itertools.combinations(seq_positions, 2):
        molten_i = seq_to_molten[seq_i]
        molten_j = seq_to_molten[seq_j]

        if not ig.get_edge_exists(molten_i, molten_j): continue
        edge = ig.find_edge(molten_i, molten_j)

        rot_index_i = get_picked_rotamer_idx(seq_i)
        rot_index_j = get_picked_rotamer_idx(seq_j)

        pair_energy = edge.get_two_body_energy(rot_index_i, rot_index_j)
        two_body_energies += pair_energy

    alt_two_body_energies = 0
    for (seq_i, seq_j), interactions in J_quadratic.items():
            for (rot_i, rot_j), e_val in interactions.items():
                k = wire_offsets[seq_i] + rot_i
                l = wire_offsets[seq_j] + rot_j

                if bitstring[k] == 1 and bitstring[l] == 1:
                    alt_two_body_energies += e_val
    two_body.append((two_body_energies, alt_two_body_energies, two_body_energies == alt_two_body_energies))

    return one_body_energies + two_body_energies
evaluate_two_energies_alt(benchmark_pose, valid_conformations, scorefxn, rotamer_lib, generator_params, ig, rot_sets, h_flex_linear, J_flex_quadratic)

Bio: -40.13631866409891 -40.13631866409891 0.0
Qua: -25.43668570369482 -25.43668570369482 0.0
Diff: -14.699632960404088

Bio: -40.13533369843573 -40.13533369843573 0.0
Qua: -25.435701869428158 -25.435701869428158 0.0
Diff: -14.699631829007572

Bio: -40.143161125379535 -40.143161125379535 0.0
Qua: -25.443528674542904 -25.443528674542904 0.0
Diff: -14.699632450836631

Bio: -40.132848311586784 -40.132848311586784 0.0
Qua: -25.433215282857418 -25.433215282857418 0.0
Diff: -14.699633028729366

Bio: -40.15793810873561 -40.15793810873561 0.0
Qua: -25.46331585198641 -25.46331585198641 0.0
Diff: -14.694622256749199

Bio: -39.912886603564445 -39.912886603564445 0.0
Qua: -25.2132550701499 -25.2132550701499 0.0
Diff: -14.699631533414546

Bio: -39.1663977020759 -39.1663977020759 0.0
Qua: -24.472052939236164 -24.472052939236164 0.0
Diff: -14.694344762839734

Bio: -38.55536369724629 -38.55536369724629 0.0
Qua: -23.871588490903378 -23.871588490903378 0.0
Diff: -14.683775206342915

Bio: -38.07965188093